# SAE Variants Analysis Pipeline - Google Colab

**Complete GPU-Optimized Training & Analysis Pipeline**

This notebook runs the COMPLETE SAE variants analysis pipeline (Phases 1-5) in Google Colab with GPU acceleration.

## Pipeline Overview
- **Phase 1:** Train 30 SAE configurations (TopK, Gated, JumpReLU, Switch) - GPU
- **Phase 2:** Configuration comparison analysis
- **Phase 2.5:** Cross-variant comparison (Pareto frontiers, interpretability)
- **Phase 2.5b:** Reconstruction fidelity analysis per variant (PCA histograms)
- **Phase 2b:** Multi-seed retraining for stability analysis
- **Phase 3a:** SAE latent space ablations (run_ablation.py)
- **Phase 3b:** Native GNN activation space ablations
- **Phase 3c:** Ablation strategy comparison
- **Phase 4:** Statistical validation
- **Phase 5:** Visualization & feature analysis

## Requirements
1. **Enable GPU:** Runtime → Change runtime type → GPU
2. **Upload project files** to Google Drive (see Cell 1)
3. **Activation data:** `outputs/activations/layer2/{train,val,test}/` required
4. **Run cells in order** (1 → 13)

## Estimated Time
- **With GPU (T4/V100):** ~7-8 hours total
- **Phase 1 (training):** ~5 hours
- **Phases 2-5 (analysis):** ~2-3 hours

---

## Cell 1: Setup Environment & Check GPU

In [ ]:
#@title 1. Check GPU & Install Dependencies

import torch
import subprocess
import sys

print("="*80)
print("SAE VARIANTS PIPELINE - GOOGLE COLAB SETUP")
print("="*80)

# Check GPU
print("\n📊 GPU CONFIGURATION")
print("-" * 80)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU Device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"✓ GPU Memory: {props.total_memory / 1e9:.1f} GB")
    print("\n✅ GPU is ready!")
else:
    print("\n⚠️  NO GPU DETECTED!")
    print("   Training will be VERY SLOW without GPU")
    print("   REQUIRED: Runtime → Change runtime type → GPU")

# Install dependencies
print("\n📦 INSTALLING DEPENDENCIES")
print("-" * 80)

packages = [
    'numpy', 'pandas', 'matplotlib', 'seaborn',
    'scipy', 'scikit-learn', 'statsmodels', 'tqdm'
]

for package in packages:
    try:
        __import__(package)
        print(f"✓ {package}")
    except ImportError:
        print(f"  Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package}")

print("\n✅ All dependencies installed!")
print("\n" + "="*80)

## Cell 2: Mount Google Drive & Load Project

In [ ]:
#@title 2. Mount Google Drive & Load Project Files

from google.colab import drive
from pathlib import Path
import shutil
import os

print("\n" + "="*80)
print("MOUNTING GOOGLE DRIVE & LOADING PROJECT")
print("="*80)

print("\n🔗 Mounting Google Drive...")
drive.mount('/content/drive')
print("✓ Google Drive mounted")

# Setup paths
DRIVE_PROJECT = Path('/content/drive/MyDrive/182-GNN_SAE')
LOCAL_PROJECT = Path('/content/project')

print(f"\n📁 Project Paths:")
print(f"   Drive:  {DRIVE_PROJECT}")
print(f"   Local:  {LOCAL_PROJECT}")

# Check if project exists
if DRIVE_PROJECT.exists():
    print(f"\n✓ Found project in Google Drive")

    # Copy to local (much faster)
    print(f"\n📋 Copying project to local storage (faster execution)...")
    if LOCAL_PROJECT.exists():
        shutil.rmtree(LOCAL_PROJECT)
    shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)
    os.chdir(LOCAL_PROJECT)
    print(f"✓ Project copied to {LOCAL_PROJECT}")

    # Verify activation data
    print(f"\n📊 CHECKING PREREQUISITE DATA")
    print("-" * 80)
    activation_dir = Path('outputs/activations/layer2')
    if activation_dir.exists():
        train_dir = activation_dir / 'train'
        val_dir = activation_dir / 'val'
        test_dir = activation_dir / 'test'
        
        train_count = len(list(train_dir.glob('*.pt'))) if train_dir.exists() else 0
        val_count = len(list(val_dir.glob('*.pt'))) if val_dir.exists() else 0
        test_count = len(list(test_dir.glob('*.pt'))) if test_dir.exists() else 0
        
        print(f"✓ Activation data found:")
        print(f"   Train: {train_count} files (graphs 0-2999)")
        print(f"   Val:   {val_count} files (graphs 3000-3499)")
        print(f"   Test:  {test_count} files (graphs 3500-3999)")
        
        if train_count == 0:
            print(f"\n⚠️  WARNING: No activation files found!")
            print(f"   Phase 1 training will FAIL without activation data")
            print(f"   Required: outputs/activations/layer2/train/*.pt")
    else:
        print(f"❌ ERROR: Activation directory not found at {activation_dir}")
        print(f"   Phase 1 training requires activation data!")
        print(f"   Please ensure outputs/activations/layer2/ is uploaded to Google Drive")

    # List Python scripts
    print(f"\n📋 CHECKING REQUIRED SCRIPTS")
    print("-" * 80)
    scripts = sorted([f for f in os.listdir('.') if f.endswith('.py')])
    required_scripts = [
        'sparse_autoencoder.py',
        'compare_sae_configs.py',
        'run_ablation.py',
        'native_gnn_ablation.py',
        'compare_ablation_strategies.py',
        'statistical_analysis_suite.py',
        'analyze_sae_reconstruction_fidelity.py'
    ]

    missing = []
    for script in required_scripts:
        if script in scripts:
            print(f"   ✓ {script}")
        else:
            print(f"   ⚠️  {script} (NOT FOUND)")
            missing.append(script)

    if missing:
        print(f"\n⚠️  WARNING: {len(missing)} required scripts missing!")
        print(f"   Make sure these are uploaded to /My Drive/182-GNN_SAE/:")
        for script in missing:
            print(f"     - {script}")

    print(f"\n✓ Current directory: {os.getcwd()}")
    print("\n✅ Project loaded successfully!")

else:
    print(f"\n❌ ERROR: Project not found at {DRIVE_PROJECT}")
    print(f"\n📌 SETUP INSTRUCTIONS:")
    print(f"   1. Create folder in Google Drive: /My Drive/182-GNN_SAE/")
    print(f"   2. Upload these REQUIRED files to that folder:")
    print(f"      - sparse_autoencoder.py")
    print(f"      - compare_sae_configs.py")
    print(f"      - run_ablation.py")
    print(f"      - native_gnn_ablation.py")
    print(f"      - compare_ablation_strategies.py")
    print(f"      - statistical_analysis_suite.py")
    print(f"      - analyze_sae_reconstruction_fidelity.py")
    print(f"   3. Upload activation data:")
    print(f"      - outputs/activations/layer2/train/*.pt (3000 files)")
    print(f"      - outputs/activations/layer2/val/*.pt (500 files)")
    print(f"      - outputs/activations/layer2/test/*.pt (500 files)")
    print(f"   4. Re-run this cell")

print("\n" + "="*80)

## PHASE 1: Train All SAE Variants (GPU-Accelerated) ⚙️

Trains 30 SAE configurations with automatic GPU acceleration:
- **TopK SAE:** 11 configs (latent_dim × k combinations)
- **Gated SAE:** 9 configs (latent_dim × sparsity_coef combinations)
- **JumpReLU SAE:** 6 configs (latent_dim × threshold_init combinations)
- **Switch SAE:** 4 configs (num_experts × latent_per_expert combinations)

**Estimated Time:** ~5 hours with GPU, ~20+ hours without GPU

In [ ]:
#@title 3. PHASE 1: Train All 30 SAE Configurations (⏱️ ~5 hours)

import subprocess
import sys
import time
import torch
from pathlib import Path
import os

print("\n" + "="*80)
print("PHASE 1: TRAINING ALL 30 SAE CONFIGURATIONS")
print("="*80)

# Verify GPU
print("\n📊 GPU Status:")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  WARNING: No GPU detected - training will be VERY SLOW")
    print("   Please use: Runtime → Change runtime type → GPU")

# Verify training script
script = Path('sparse_autoencoder.py')
if not script.exists():
    print(f"\n❌ ERROR: {script} not found!")
    print(f"   Verify all files are uploaded to Google Drive")
else:
    print(f"\n✓ Found training script: {script}")

    # Check activation data
    activation_dir = Path('outputs/activations/layer2/train')
    if activation_dir.exists():
        act_files = len(list(activation_dir.glob('*.pt')))
        print(f"✓ Found activation data: {act_files} training files")
    else:
        print(f"❌ ERROR: Activation directory not found at {activation_dir}")
        print(f"   Training will FAIL without activation data!")

    # Create checkpoints directory
    Path('checkpoints').mkdir(exist_ok=True)
    Path('outputs').mkdir(exist_ok=True)

    print(f"\n{'='*80}")
    print("📝 TRAINING CONFIGURATION")
    print(f"{'='*80}")
    print(f"Total Configurations: 30 (11 TopK + 9 Gated + 6 JumpReLU + 4 Switch)")
    print(f"Start Time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"GPU Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (NOT RECOMMENDED)'}")
    print(f"\nThis will take approximately 5 hours with GPU")
    print(f"Do NOT interrupt the kernel while training is in progress")
    print(f"\n{'='*80}\n")

    # Run training
    result = subprocess.run(
        [sys.executable, str(script)],
        capture_output=False,
        text=True
    )

    print(f"\n{'='*80}")
    print(f"End Time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}")

    # Report results
    if result.returncode == 0:
        checkpoints = list(Path('checkpoints').glob('sae_*.pt'))
        metrics = list(Path('outputs').glob('sae_metrics_*.json'))

        print(f"\n✅ TRAINING COMPLETED SUCCESSFULLY!")
        print(f"\n📊 Results:")
        print(f"   Total Checkpoints: {len(checkpoints)}/30")
        print(f"   Total Metrics Files: {len(metrics)}/30")

        # Breakdown by variant
        print(f"\n   Variant Breakdown:")
        topk = len(list(Path('checkpoints').glob('sae_topk_*.pt')))
        gated = len(list(Path('checkpoints').glob('sae_gated_*.pt')))
        jumprelu = len(list(Path('checkpoints').glob('sae_jumprelu_*.pt')))
        switch = len(list(Path('checkpoints').glob('sae_switch_*.pt')))

        print(f"      TopK:     {topk}/11 {'✓' if topk == 11 else ''}")
        print(f"      Gated:    {gated}/9 {'✓' if gated == 9 else ''}")
        print(f"      JumpReLU: {jumprelu}/6 {'✓' if jumprelu == 6 else ''}")
        print(f"      Switch:   {switch}/4 {'✓' if switch == 4 else ''}")

# VALIDATION: Verify all 30 checkpoints exist
print(f"\n{'─'*80}")
print(f"🔍 VALIDATION: Checkpoint Count")
print(f"{'─'*80}")

total_variants = topk + gated + jumprelu + switch

if len(checkpoints) == 30 and total_variants == 30:
    print(f"✓ SUCCESS: All 30 checkpoints verified!")
else:
    print(f"⚠️  WARNING: Checkpoint count mismatch!")
    print(f"   Expected: 30 total checkpoints")
    print(f"   Found: {len(checkpoints)} checkpoints")
    print(f"   By variant: {topk} + {gated} + {jumprelu} + {switch} = {total_variants}")
    
    if len(checkpoints) < 30:
        print(f"\n   ⚠️  {30 - len(checkpoints)} checkpoint(s) missing!")
        print(f"   Some configurations may have failed during training")
        print(f"   Review the training output above for error messages")



        print(f"\n✓ All training data saved to checkpoints/ and outputs/")
        print(f"\n→ Continue to PHASE 2")
    else:
        print(f"\n❌ Training failed with return code {result.returncode}")
        print(f"   Check error messages above for details")

In [ ]:
#@title 3b. Monitor Training Progress (Run periodically during Phase 1)

import json
from pathlib import Path

print("\n" + "="*80)
print("TRAINING PROGRESS MONITOR")
print("="*80)

ckpt_dir = Path('checkpoints')

if not ckpt_dir.exists() or len(list(ckpt_dir.glob('*.pt'))) == 0:
    print("\n⏳ No checkpoints yet - training hasn't started or is in progress")
else:
    checkpoints = list(ckpt_dir.glob('sae_*.pt'))
    metrics_dir = Path('outputs')
    metrics = list(metrics_dir.glob('sae_metrics_*.json'))

    print(f"\n📈 OVERALL PROGRESS: {len(checkpoints)}/30 configurations trained")

    # Variant breakdown with progress bars
    print(f"\n   Variant Status:")
    variants = {
        'topk': ('sae_topk_*.pt', 11),
        'gated': ('sae_gated_*.pt', 9),
        'jumprelu': ('sae_jumprelu_*.pt', 6),
        'switch': ('sae_switch_*.pt', 4)
    }

    for variant, (pattern, expected) in variants.items():
        count = len(list(ckpt_dir.glob(pattern)))
        pct = int(100 * count / expected)
        bar_length = 20
        filled = int(bar_length * count / expected)
        bar = "█" * filled + "░" * (bar_length - filled)
        status = "✓" if count == expected else "⏳"
        print(f"   {variant.upper():10} [{bar}] {count}/{expected} ({pct}%) {status}")

    # Latest trained config info
    if len(metrics) > 0:
        latest_metric = sorted(metrics)[-1]
        print(f"\n   Most Recently Trained Config: {latest_metric.name}")

        try:
            with open(latest_metric, 'r') as f:
                data = json.load(f)

            if 'best_epoch' in data:
                print(f"      Best Epoch: {data['best_epoch']}")

            if 'test_metrics' in data and 'mse' in data['test_metrics']:
                print(f"      Test MSE: {data['test_metrics']['mse']:.6f}")

            if 'training_time' in data:
                hours = data['training_time'] / 3600
                print(f"      Training Time: {hours:.1f} hours")
        except:
            pass

print(f"\n" + "="*80)

## PHASE 2: Configuration Comparison 📊

Analyzes all 30 configurations and creates comparison:
- Within-variant ranking by composite score (reconstruction × sparsity × interpretability)
- Feature-motif correlation (point-biserial r_pb) analysis
- Best configuration identification per variant
- Test set definition (test_graph_ids.json)

**Time:** ~15 minutes

In [ ]:
#@title 4. PHASE 2: Configuration Comparison (~15 min)

import subprocess
import sys
from pathlib import Path

print("\n" + "="*80)
print("PHASE 2: CONFIGURATION COMPARISON")
print("="*80)

script = Path('compare_sae_configs.py')

if not script.exists():
    print(f"\n❌ ERROR: {script} not found!")
else:
    print(f"\n✓ Found analysis script: {script}")
    print(f"\n📊 Running within-variant comparison...")
    print(f"   - Ranking configs by composite_score")
    print(f"   - Computing feature-motif correlations (r_pb)")
    print(f"   - Generating test_graph_ids.json for Phase 3\n")

    result = subprocess.run(
        [sys.executable, str(script)],
        capture_output=False
    )

    if result.returncode == 0:
        print(f"\n\n{'='*80}")
        print(f"✅ PHASE 2 COMPLETED SUCCESSFULLY!")
        print(f"{'='*80}")

        # Verify outputs
        csv_file = Path('outputs/sae_config_comparison.csv')
        test_ids = Path('outputs/test_graph_ids.json')
        
        if csv_file.exists():
            import pandas as pd
            df = pd.read_csv(csv_file)
            print(f"\n📊 Configuration Comparison Results:")
            print(f"   Configurations analyzed: {len(df)}")
            
            # Show best per variant
            print(f"\n   Best Config Per Variant:")
            for variant in ['topk', 'gated', 'jumprelu', 'switch']:
                v_data = df[df['variant'] == variant]
                if len(v_data) > 0:
                    best = v_data.iloc[0]
                    print(f"      {variant.upper()}: {best['config_name']} (score: {best['composite_score']:.3f})")
        
        if test_ids.exists():
            import json
            with open(test_ids) as f:
                test_data = json.load(f)
            print(f"\n   Test Set Definition:")
            print(f"      Total test graphs: {len(test_data)}")
        
        print(f"\n✓ Outputs saved to: outputs/sae_config_comparison.csv, outputs/test_graph_ids.json")
        print(f"\n→ Continue to PHASE 3a")
    else:
        print(f"\n❌ Phase 2 failed with return code {result.returncode}")

## PHASE 2.5: Cross-Variant Comparison 📊

Systematic comparison across all SAE variants (TopK, Gated, JumpReLU, Switch):
- Pareto frontiers: reconstruction quality vs sparsity trade-offs
- Cross-variant metrics: interpretability, efficiency, reconstruction fidelity
- Comparative visualizations and summary report

**Purpose:** Understand which architecture achieves best overall performance

**Time:** ~3 minutes

In [ ]:
#@title 4a. PHASE 2.5: Cross-Variant Comparison (~3 min)

import subprocess
import sys
from pathlib import Path

print("\n" + "="*80)
print("PHASE 2.5: CROSS-VARIANT COMPARISON")
print("="*80)

script = Path('compare_sae_variants.py')

if not script.exists():
    print(f"\n❌ ERROR: {script} not found!")
else:
    print(f"\n✓ Found cross-variant comparison script: {script}")

    # Check Phase 1 prerequisites
    checkpoints = list(Path('checkpoints').glob('sae_*.pt'))
    metrics = list(Path('outputs').glob('sae_metrics_*.json'))

    if len(checkpoints) == 0 or len(metrics) == 0:
        print(f"\n❌ ERROR: Phase 1 outputs not found!")
        print(f"   Checkpoints: {len(checkpoints)}/30")
        print(f"   Metrics: {len(metrics)}/30")
        print(f"   CRITICAL: Phase 1 (sparse_autoencoder.py) must complete first")
    else:
        print(f"\n✓ Phase 1 outputs verified:")
        print(f"   Checkpoints: {len(checkpoints)}/30")
        print(f"   Metrics: {len(metrics)}/30")

        print(f"\n📊 Running cross-variant comparison...")
        print(f"   - Pareto frontier: Reconstruction MSE vs Sparsity")
        print(f"   - Interpretability metrics per variant")
        print(f"   - Computational efficiency comparison")
        print(f"   - Summary report\n")

        result = subprocess.run(
            [sys.executable, str(script)],
            capture_output=False
        )

        if result.returncode == 0:
            print(f"\n{'='*80}")
            print(f"✅ PHASE 2.5 COMPLETED SUCCESSFULLY!")
            print(f"{'='*80}")

            # Verify outputs
            comparison_csv = Path('outputs/sae_variant_comparison.csv')
            variant_plots = Path('outputs/variant_comparison_plots')
            report = Path('outputs/variant_comparison_report.md')

            print(f"\n📊 Generated Outputs:")
            if comparison_csv.exists():
                print(f"   ✓ CSV: sae_variant_comparison.csv")
            if variant_plots.exists():
                plots = list(variant_plots.glob('*.png'))
                print(f"   ✓ Plots: {len(plots)} visualizations")
                for plot in sorted(plots):
                    print(f"      - {plot.name}")
            if report.exists():
                print(f"   ✓ Report: variant_comparison_report.md")

            print(f"\n✓ Results saved to: outputs/")
            print(f"\n→ Continue to PHASE 2.5b (Reconstruction Fidelity per Variant)")
        else:
            print(f"\n❌ Phase 2.5 failed with return code {result.returncode}")
            print(f"   ⚠️  Phase 2.5 is optional - continue to Phase 2.5b")

## PHASE 2.5b: Reconstruction Fidelity Analysis per Variant 📊

Analyzes how faithfully each SAE variant reconstructs native activations using PCA component histograms:
- **TopK best config:** PCA fidelity for TopK's best configuration
- **Gated best config:** PCA fidelity for Gated's best configuration  
- **JumpReLU best config:** PCA fidelity for JumpReLU's best configuration
- **Switch best config:** PCA fidelity for Switch's best configuration

**Key Insight:** Compares distribution preservation across variants. Low correlation between native and reconstructed PCA components indicates the variant misses important variance structure.

**DEPENDS ON:** Phase 2 results (best config per variant identified)

**Time:** ~10 minutes

In [ ]:
#@title 4c. PHASE 2.5b: Reconstruction Fidelity per Variant (~10 min)

import subprocess
import sys
from pathlib import Path
import pandas as pd
import json

print("\n" + "="*80)
print("PHASE 2.5b: RECONSTRUCTION FIDELITY ANALYSIS PER VARIANT")
print("="*80)

script = Path('analyze_sae_reconstruction_fidelity.py')

if not script.exists():
    print(f"\n❌ ERROR: {script} not found!")
else:
    print(f"\n✓ Found reconstruction fidelity script: {script}")

    # Load best configs from Phase 2
    csv_file = Path('outputs/sae_config_comparison.csv')
    if not csv_file.exists():
        print(f"\n❌ ERROR: {csv_file} not found!")
        print(f"   CRITICAL: Phase 2 must complete before Phase 2.5b")
    else:
        df = pd.read_csv(csv_file)
        
        print(f"\n✓ Loading best configurations per variant from Phase 2:")
        best_configs = {}
        
        for variant in ['topk', 'gated', 'jumprelu', 'switch']:
            v_data = df[df['variant'] == variant]
            if len(v_data) > 0:
                best = v_data.iloc[0]
                best_configs[variant] = best
                print(f"   ✓ {variant.upper():10} | latent_dim={int(best['latent_dim']):3d} | score={best['composite_score']:.3f}")
        
        # Create output directory
        output_dir = Path('outputs/sae_reconstruction_fidelity')
        output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\n📊 Running PCA reconstruction fidelity analysis for each variant...")
        print(f"   This generates separate PCA histograms comparing native vs reconstructed activations")
        print(f"   Output: 4 separate plots (one per variant)\n")
        
        variant_metrics = {}
        
        # Run analysis for each variant
        for variant, best_config in best_configs.items():
            print(f"\n{'─'*80}")
            print(f"Analyzing: {variant.upper()}")
            print(f"{'─'*80}")
            
            # Build command based on variant
            if variant == 'topk':
                cmd = [
                    sys.executable, str(script),
                    '--variant', 'topk',
                    '--latent-dim', str(int(best_config['latent_dim'])),
                    '--k', str(int(best_config['k'])),
                    '--num-graphs', '100',
                    '--output-dir', str(output_dir)
                ]
            elif variant == 'gated':
                cmd = [
                    sys.executable, str(script),
                    '--variant', 'gated',
                    '--latent-dim', str(int(best_config['latent_dim'])),
                    '--sparsity-coef', str(best_config['sparsity_coef']),
                    '--num-graphs', '100',
                    '--output-dir', str(output_dir)
                ]
            elif variant == 'jumprelu':
                cmd = [
                    sys.executable, str(script),
                    '--variant', 'jumprelu',
                    '--latent-dim', str(int(best_config['latent_dim'])),
                    '--threshold-init', str(best_config['threshold_init']),
                    '--bandwidth', '0.01',
                    '--num-graphs', '100',
                    '--output-dir', str(output_dir)
                ]
            elif variant == 'switch':
                cmd = [
                    sys.executable, str(script),
                    '--variant', 'switch',
                    '--num-experts', str(int(best_config['num_experts'])),
                    '--latent-per-expert', str(int(best_config['latent_per_expert'])),
                    '--k-per-expert', str(int(best_config['k_per_expert'])),
                    '--num-graphs', '100',
                    '--output-dir', str(output_dir)
                ]
            
            result = subprocess.run(cmd, capture_output=False)
            
            if result.returncode == 0:
                print(f"\n✓ {variant.upper()} analysis completed")
                
                # Try to load metrics
                metrics_file = output_dir / f'metrics_{variant}.json'
                if metrics_file.exists():
                    try:
                        with open(metrics_file, 'r') as f:
                            metrics = json.load(f)
                        variant_metrics[variant] = metrics
                    except:
                        pass
            else:
                print(f"\n⚠️  {variant.upper()} analysis failed (attempting next variant)")
        
        print(f"\n\n{'='*80}")
        print(f"✅ PHASE 2.5b COMPLETED!")
        print(f"{'='*80}")
        
        # Show summary comparison
        if len(variant_metrics) > 0:
            print(f"\n📊 Reconstruction Fidelity Summary Across Variants:")
            print(f"{'─'*80}\n")
            
            summary_data = []
            for variant, metrics in variant_metrics.items():
                summary_data.append({
                    'Variant': variant.upper(),
                    'MSE': f"{metrics.get('mse', 0):.6f}",
                    'RMSE': f"{metrics.get('rmse', 0):.6f}",
                    'Var Ratio': f"{metrics.get('variance_ratio', 0):.4f}",
                    'Mean Diff': f"{metrics.get('mean_diff', 0):.6f}",
                    'Std Diff': f"{metrics.get('std_diff', 0):.6f}"
                })
            
            df_summary = pd.DataFrame(summary_data)
            print(df_summary.to_string(index=False))
            
            # Interpretation guide
            print(f"\n{'─'*80}")
            print(f"Interpretation:")
            print(f"  MSE: Lower is better (point-wise reconstruction error)")
            print(f"  Var Ratio: Close to 1.0 is better (variance preservation)")
            print(f"  Mean Diff: Smaller indicates better mean preservation")
            print(f"  Std Diff: Smaller indicates better spread preservation")
            
            # Find best variants
            mse_values = {v: float(m['MSE']) for v, m in zip([d['Variant'] for d in summary_data], variant_metrics.values())}
            best_mse = min(mse_values, key=mse_values.get)
            print(f"\n  Best MSE: {best_mse}")
        
        # Verify output files
        plots = list(output_dir.glob('pca_histograms_*.png'))
        print(f"\n📈 Generated Plots:")
        print(f"   {len(plots)}/4 variants")
        for plot in sorted(plots):
            print(f"   ✓ {plot.name}")
        
        print(f"\n✓ Results saved to: {output_dir}/")
        print(f"\n→ Continue to PHASE 2b (Multi-seed Retraining)")

## PHASE 2b: Retrain Best Configs with Multiple Seeds 🔄

Multi-seed training for reproducibility and stability analysis:
- Retrains best configuration per variant (TopK, Gated, JumpReLU, Switch)
- Uses 5 seeds: [42, 123, 456, 789, 1011] for statistical validation
- Generates multi-seed checkpoints for Phase 4 stability analysis
- Computes feature stability metrics (decoder similarity across seeds)

**CRITICAL FOR PUBLICATION:** Phase 4 statistical analysis requires these multi-seed models

**Time:** ~2.7 hours (16 additional training runs)

In [ ]:
#@title 4b. PHASE 2b: Retrain Best Configs with Multiple Seeds (~2.7 hours)

import subprocess
import sys
from pathlib import Path
import pandas as pd

print("\n" + "="*80)
print("PHASE 2b: MULTI-SEED TRAINING FOR BEST CONFIGURATIONS")
print("="*80)

script = Path('retrain_best_configs.py')

if not script.exists():
    print(f"\n❌ ERROR: {script} not found!")
    print(f"   CRITICAL: Phase 4 statistical analysis requires multi-seed models")
else:
    print(f"\n✓ Found retraining script: {script}")

    # Get best config from Phase 2
    csv_file = Path('outputs/sae_config_comparison.csv')
    if not csv_file.exists():
        print(f"\n❌ ERROR: {csv_file} not found!")
        print(f"   CRITICAL: Phase 2 must complete before Phase 2b")
    else:
        df = pd.read_csv(csv_file)
        best_configs = {}
        
        print(f"\n📊 Best Configurations Identified in Phase 2:")
        for variant in ['topk', 'gated', 'jumprelu', 'switch']:
            v_data = df[df['variant'] == variant]
            if len(v_data) > 0:
                best = v_data.iloc[0]
                print(f"   ✓ {variant.upper():10} | {best['config_name']}")
                best_configs[variant] = best
        
        seeds = [42, 123, 456, 789, 1011]
        
        print(f"\n🔄 MULTI-SEED RETRAINING")
        print(f"{'─'*80}")
        print(f"Seeds: {seeds}")
        print(f"Total Training Runs: {len(best_configs)} variants × {len(seeds)} seeds = {len(best_configs) * len(seeds)} runs")
        print(f"Status: Original seed=42 models already trained in Phase 1")
        print(f"Action: Retrain each best config with seeds [123, 456, 789, 1011]")
        print(f"\nOutput: Multi-seed checkpoints")
        print(f"   checkpoints/sae_topk_latent512_k8_seed42.pt         (already exists)")
        print(f"   checkpoints/sae_topk_latent512_k8_seed123.pt       (new)")
        print(f"   checkpoints/sae_topk_latent512_k8_seed456.pt       (new)")
        print(f"   checkpoints/sae_topk_latent512_k8_seed789.pt       (new)")
        print(f"   checkpoints/sae_topk_latent512_k8_seed1011.pt      (new)")
        print(f"   ... (same for gated, jumprelu, switch)\n")

        print(f"🚀 Starting multi-seed retraining...\n")

        result = subprocess.run(
            [sys.executable, str(script),
             '--seeds', '123', '456', '789', '1011'],
            capture_output=False
        )

        if result.returncode == 0:
            print(f"\n{'='*80}")
            print(f"✅ PHASE 2b COMPLETED SUCCESSFULLY!")
            print(f"{'='*80}")

            # Verify multi-seed checkpoints
            import glob
            all_ckpts = list(Path('checkpoints').glob('sae_*_seed*.pt'))
            print(f"\n📊 Multi-Seed Checkpoint Summary:")
            print(f"   Total multi-seed checkpoints: {len(all_ckpts)}")

            # Count by variant
            for variant in ['topk', 'gated', 'jumprelu', 'switch']:
                variant_ckpts = len(list(Path('checkpoints').glob(f'sae_{variant}_*_seed*.pt')))
                print(f"   {variant.upper():10} multi-seed: {variant_ckpts}/5 seeds")

            # Check summary file
            summary_file = Path('outputs/retrain_summary.json')
            if summary_file.exists():
                import json
                with open(summary_file) as f:
                    summary = json.load(f)
                print(f"\n📈 Stability Metrics (across seeds):")
                for variant, stats in summary.items():
                    mse_mean = stats['test_mse']['mean']
                    mse_std = stats['test_mse']['std']
                    cv = stats['test_mse']['cv']
                    print(f"   {variant.upper():10} | MSE: {mse_mean:.6f} ± {mse_std:.6f} (CV: {cv:.3f})")

            print(f"\n✓ Multi-seed models ready for Phase 4 stability analysis")
            print(f"\n→ Continue to PHASE 3a (SAE Latent Ablations)")
        else:
            print(f"\n❌ Phase 2b failed with return code {result.returncode}")
            print(f"   ⚠️  Phase 4 statistical analysis will be limited without multi-seed models")

## OPTIONAL Phase 2B+: Nested Validation - Multi-Seed Stability Analysis 🔄

Validates feature reproducibility across seeds:
- **Purpose:** Address reviewer concern about selection bias in model selection
- **Method:** Compute decoder weight similarity across 5 seeds [42, 123, 456, 789, 1011]
- **Output:** Stability reports + feature reproducibility analysis
- **Status:** OPTIONAL - All subsequent phases run fine without this

**Note:** This analysis confirms that selected features are stable across random seeds, not artifacts of a single initialization.

**Time:** ~10 minutes

In [ ]:
#@title 4b+. OPTIONAL: Nested Validation - Multi-Seed Stability Report (~10 min)

import subprocess
import sys
from pathlib import Path

print("\n" + "="*80)
print("OPTIONAL: NESTED VALIDATION - MULTI-SEED STABILITY ANALYSIS")
print("="*80)

print("\n📌 This analysis is OPTIONAL and validates stability across seeds")
print("   (All subsequent phases will run fine without it)")

script = Path('aggregate_validation_report.py')

if not script.exists():
    print(f"\n⚠️  {script} not found - skipping optional analysis")
else:
    print(f"\n✓ Found validation report script: {script}")

    # Check if Phase 2b multi-seed checkpoints exist
    multi_seed_ckpts = list(Path('checkpoints').glob('sae_*_seed*.pt'))
    
    if len(multi_seed_ckpts) < 5:
        print(f"\n⚠️  Skipping: Multi-seed checkpoints not found")
        print(f"   Expected: Phase 2b should have created sae_*_seed*.pt files")
        print(f"   Found: {len(multi_seed_ckpts)} multi-seed checkpoints")
    else:
        print(f"\n✓ Found {len(multi_seed_ckpts)} multi-seed checkpoints")
        print(f"\n📊 Running stability analysis across all 4 variants...")
        print(f"   Analyzing feature consistency across seeds [42, 123, 456, 789, 1011]")
        print(f"   Computing decoder weight similarity (feature reproducibility)\n")

        result = subprocess.run([
            sys.executable, str(script),
            '--all-variants'
        ], capture_output=False)

        if result.returncode == 0:
            print(f"\n{'='*80}")
            print(f"✅ NESTED VALIDATION REPORT COMPLETE!")
            print(f"{'='*80}")

            # Verify outputs
            stability_dir = Path('outputs/stability_analysis')
            if stability_dir.exists():
                reports = list(stability_dir.glob('nested_validation_*.md'))
                plots = list(stability_dir.glob('feature_stability_*.png'))

                print(f"\n📊 Generated Outputs:")
                print(f"   Markdown reports: {len(reports)}")
                for report in sorted(reports):
                    print(f"      ✓ {report.name}")

                print(f"\n   Visualization plots: {len(plots)}")
                for plot in sorted(plots):
                    print(f"      ✓ {plot.name}")

                print(f"\n   Summary:")
                print(f"   - Feature stability metrics per variant")
                print(f"   - Decoder weight reproducibility across seeds")
                print(f"   - Addresses reviewer concern about selection bias")

            print(f"\n✓ Results saved to: outputs/stability_analysis/")
        else:
            print(f"\n⚠️  Nested validation failed (non-critical - skipping)")

print(f"\n→ Continue to PHASE 2.5b (Feature Significance Analysis)")

## PHASE 3a: SAE Latent Space Ablations 🔬

Three-way comparison of GNN loss:
1. **Original:** GNN loss on native 64D activations (no SAE)
2. **Full SAE:** GNN loss on SAE reconstruction
3. **Ablated:** GNN loss on SAE reconstruction with feature zeroed

**CRITICAL:** This phase selects the overall best SAE configuration to use for all downstream analysis (Phases 3b-5). The ranking metric from Phase 2 determines which config is selected.

**Config Selection Strategy:**
- **Default metric (max_rpb_abs):** Selects config with strongest feature-motif correlations
  - Recommended: Direct, interpretable measure of alignment quality
  - Command: `python compare_sae_configs.py` (or explicit `--metric max_rpb_abs`)
- **Alternative metric (composite_score):** Balances effect size + predictive power + efficiency  
  - Use if: Want to optimize for multiple factors simultaneously
  - Command: `python compare_sae_configs.py --metric composite_score`

Once Phase 2 completes with your chosen metric, Phase 3a automatically selects the top-ranked config from `outputs/sae_config_comparison.csv`.


Measures feature importance via ablation impact = Loss(Ablated) - Loss(Full SAE)

**CRITICAL:** Generates `ablations/results/ablation_*.csv` files required by Phase 3c

**Time:** ~2 hours

In [ ]:
#@title 5. PHASE 3a: Motif-Guided SAE Latent Ablations (~2 hours)

import subprocess
import sys
from pathlib import Path
import pandas as pd
import json

print("\\n" + "="*80)
print("PHASE 3a: MOTIF-GUIDED SAE LATENT SPACE ABLATIONS")
print("="*80)

script = Path('run_interpretability_experiments.py')

if not script.exists():
    print(f"\\n❌ ERROR: {script} not found!")
else:
    print(f"\\n✓ Found motif-guided ablation script: {script}")

    # Load Phase 2 results to get best config
    csv_file = Path('outputs/sae_config_comparison.csv')
    if not csv_file.exists():
        print(f"\\n❌ ERROR: {csv_file} not found!")
        print(f"   CRITICAL: Phase 2 (compare_sae_configs.py) must complete first")
        sys.exit(1)

    df = pd.read_csv(csv_file)

    # Select best config by MAX_RPB_ABS (not composite_score)
    best_idx = df['max_rpb_abs'].idxmax()
    best = df.loc[best_idx]

    best_variant = best['variant']
    best_latent = int(best['latent_dim'])

    print(f"\\n✓ Selected best config by MAX POINT-BISERIAL CORRELATION:")
    print(f"   Variant: {best_variant.upper()}")
    print(f"   Latent Dim: {best_latent}")

    # Extract variant-specific parameters
    if best_variant == 'topk':
        best_k = int(best['k'])
        param_str = f"latent_dim={best_latent}, k={best_k}"
        params = {'variant': best_variant, 'latent_dim': best_latent, 'k': best_k}
    elif best_variant == 'gated':
        best_sparsity_coef = float(best['sparsity_coef'])
        param_str = f"latent_dim={best_latent}, sparsity_coef={best_sparsity_coef:.0e}"
        params = {'variant': best_variant, 'latent_dim': best_latent, 'sparsity_coef': best_sparsity_coef}
    elif best_variant == 'jumprelu':
        best_threshold = float(best['threshold_init'])
        param_str = f"latent_dim={best_latent}, threshold_init={best_threshold:.0e}"
        params = {'variant': best_variant, 'latent_dim': best_latent, 'threshold_init': best_threshold}
    elif best_variant == 'switch':
        best_num_experts = int(best['num_experts'])
        best_latent_per_expert = int(best['latent_per_expert'])
        best_k_per_expert = int(best['k_per_expert'])
        param_str = f"num_experts={best_num_experts}, latent_per_expert={best_latent_per_expert}, k_per_expert={best_k_per_expert}"
        params = {
            'variant': best_variant,
            'num_experts': best_num_experts,
            'latent_per_expert': best_latent_per_expert,
            'k_per_expert': best_k_per_expert
        }

    print(f"   {param_str}")
    print(f"   Max |rpb|: {best['max_rpb_abs']:.3f}")

    # Save metadata for Phase 3b and 3c to use
    metadata = {
        'phase': '3a',
        'selected_by': 'max_rpb_abs',
        'best_config': params,
        'max_rpb_abs': float(best['max_rpb_abs']),
        'composite_score': float(best['composite_score'])
    }

    ablations_dir = Path('ablations')
    ablations_dir.mkdir(exist_ok=True)

    metadata_file = ablations_dir / 'phase_3a_config.json'
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"\\n✓ Saved config metadata to: {metadata_file}")

    # EXPLICIT CONFIRMATION: This config will be used for ALL remaining phases
    print(f"\n{'='*80}")
    print(f"✅ BEST CONFIG SELECTED FOR PIPELINE")
    print(f"{'='*80}")
    print(f"The following configuration will be used for:")
    print(f"  • Phase 3a: SAE latent space ablations (THIS PHASE)")
    print(f"  • Phase 3b: Native GNN ablations (will analyze same features)")
    print(f"  • Phase 3c: Ablation strategy comparison")
    print(f"  • Phase 3d: Mixed-motif robustness testing")
    print(f"\nConfiguration:")
    print(f"  Variant: {best_variant.upper()}")
    print(f"  Latent Dim: {best_latent}")
    print(f"  {param_str}")
    print(f"  Max |rpb|: {best['max_rpb_abs']:.3f}")
    print(f"{'='*80}\n")


    print(f"\\n🔍 Running motif-guided SAE latent space ablations...")
    print(f"   Process: Ablate top features for each motif group")
    print(f"   Statistics: z-scores, percentiles, p-values vs random controls")
    print(f"   Output: ablations/results/ (grouped motif results)")
    print(f"   Also: ablations/interpretability_*/ (aggregated stats)\\n")

    # Run with variant parameter
    cmd = [
        sys.executable, str(script),
        '--variant', best_variant,
        '--latent_dim', str(best_latent),
        '--min_rpb', '0.05',  # Configurable threshold
        '--n_random_trials', '20'  # Configurable trials
    ]

    # Add variant-specific parameters to command if needed
    if best_variant == 'topk':
        cmd.extend(['--k', str(best_k)])

    result = subprocess.run(cmd, capture_output=False)

    if result.returncode == 0:
        print(f"\\n\\n{'='*80}")
        print(f"✅ PHASE 3a COMPLETED SUCCESSFULLY!")
        print(f"{'='*80}")

        # Verify outputs
        ablation_results = Path('ablations/results')
        if ablation_results.exists():
            result_files = list(ablation_results.glob('*.csv'))
            print(f"\\n📊 Ablation Results:")
            print(f"   Total result files: {len(result_files)}")

            # Show motif results
            motif_names = ['feedback_loop', 'cascade', 'feedforward_loop', 'single_input_module']
            print(f"\\n   Motif-Specific Results (by variant {best_variant.upper()}):")
            for motif in motif_names:
                motif_files = [f for f in result_files if motif in f.name]
                status = "✓" if len(motif_files) > 0 else "⚠️ "
                print(f"   {status} {motif}: {len(motif_files)} file(s)")
                for f in motif_files:
                    print(f"      - {f.name}")

        print(f"\\n✓ Outputs are ready for Phase 3b and 3c")
        print(f"\\n✓ Config metadata saved to: {metadata_file}")
        print(f"\\n→ Continue to PHASE 3b")
    else:
        print(f"\\n❌ Phase 3a failed with return code {result.returncode}")
        print(f"   CRITICAL: Phase 3b and 3c will fail without Phase 3a outputs!")

## PHASE 3b: Native GNN Activation Space Ablations 🧪

Direct activation patching (no SAE reconstruction):
- Identifies nodes most activated by each SAE feature
- Patches (zeros) their activations in native 64D GNN space
- Measures GNN impact directly
- Conditional analysis: with/without motif presence

**Time:** ~1 hour

In [ ]:
#@title 6. PHASE 3b: Native GNN Ablations (~1 hour for all 4 motifs)

import subprocess
import sys
from pathlib import Path
import pandas as pd
import json

print("\\n" + "="*80)
print("PHASE 3b: NATIVE GNN ACTIVATION SPACE ABLATIONS (ALL 4 MOTIFS)")
print("="*80)

script = Path('native_gnn_ablation.py')

if not script.exists():
    print(f"\\n❌ ERROR: {script} not found!")
else:
    print(f"\\n✓ Found ablation script: {script}")

    # Get best config from Phase 3a metadata
    metadata_file = Path('ablations/phase_3a_config.json')
    if not metadata_file.exists():
        print(f"\\n❌ ERROR: Phase 3a metadata not found!")
        print(f"   Please run Phase 3a first to generate: {metadata_file}")
        sys.exit(1)

    with open(metadata_file, 'r') as f:
        metadata = json.load(f)

    best_config = metadata['best_config']
    best_variant = best_config['variant']
    best_latent = best_config['latent_dim']

    print(f"\\n✓ Using best config from Phase 3a:")
    print(f"   Variant: {best_variant}")
    print(f"   Latent Dim: {best_latent}")

    # Extract variant-specific parameters
    if best_variant == 'topk':
        best_k = int(best_config['k'])
        print(f"   K: {best_k}")
    elif best_variant == 'gated':
        best_k = int(best_config.get('k', 8))  # Use default or from config
        print(f"   Sparsity Coef: {best_config['sparsity_coef']:.0e}")
    elif best_variant == 'jumprelu':
        best_k = 8  # JumpReLU doesn't use k
        print(f"   Threshold: {best_config['threshold_init']:.0e}")
    elif best_variant == 'switch':
        best_k = int(best_config['k_per_expert'])
        print(f"   Num Experts: {best_config['num_experts']}")
        print(f"   Latent per Expert: {best_config['latent_per_expert']}")
        print(f"   K per Expert: {best_config['k_per_expert']}")

    print(f"\\n🔍 Running native GNN activation space ablations...")
    print(f"   Strategy: SAE r_pb-guided node patching per motif")
    print(f"   Method: Direct activation patching in 64D space")
    print(f"   Processing: All 4 motifs separately")
    print(f"     1. in_feedback_loop")
    print(f"     2. in_cascade")
    print(f"     3. in_feedforward_loop")
    print(f"     4. in_single_input_module")
    print(f"   Output: outputs/native_gnn_ablations/ (REQUIRED for Phase 3c)\\n")

    # Define all 4 motifs
    motifs = [
        'in_feedback_loop',
        'in_cascade',
        'in_feedforward_loop',
        'in_single_input_module'
    ]

    results_count = 0
    failed_motifs = []

    # Run native ablation for EACH motif SEPARATELY
    for i, motif in enumerate(motifs, 1):
        print(f"\\n{'─'*80}")
        print(f"[{i}/{len(motifs)}] Processing Motif: {motif}")
        print(f"{'─'*80}")
        print(f"Running: native_gnn_ablation.py --variant {best_variant} --latent_dim {best_latent} --use-rpb --motif {motif}\\n")

        result = subprocess.run(
            [sys.executable, str(script),
             '--variant', best_variant,
             '--latent_dim', str(best_latent),
             '--use-rpb',
             '--motif', motif],
            capture_output=False
        )

        if result.returncode == 0:
            results_count += 1
            print(f"\\n✓ [{i}/{len(motifs)}] {motif} completed successfully")
        else:
            failed_motifs.append(motif)
            print(f"\\n⚠️  [{i}/{len(motifs)}] {motif} failed (return code {result.returncode})")

    print(f"\\n\\n{'='*80}")
    if results_count == len(motifs):
        print(f"✅ PHASE 3b COMPLETED SUCCESSFULLY!")
        print(f"{'='*80}")
        print(f"\\n✓ All 4 motifs processed successfully")

        # Verify outputs
        ablation_dir = Path('outputs/native_gnn_ablations')
        if ablation_dir.exists():
            results = list(ablation_dir.glob('native_ablation_*.csv'))
            print(f"\\n📊 Native Ablation Results:")
            print(f"   Total result files: {len(results)}")

            print(f"\\n   Motif-Specific Results:")
            for motif in motifs:
                motif_files = [f for f in results if motif in f.name]
                status = "✓" if len(motif_files) > 0 else "⚠️ "
                print(f"   {status} {motif}: {len(motif_files)} file(s)")
                for f in motif_files:
                    print(f"      - {f.name}")

            print(f"\\n✓ Outputs are ready for Phase 3c (motif-grouped comparison)")

        print(f"\\n✓ Results saved to: outputs/native_gnn_ablations/")
        print(f"\\n→ Continue to PHASE 3c")
    else:
        print(f"⚠️  PHASE 3b PARTIALLY COMPLETED")
        print(f"{'='*80}")
        print(f"\\n✓ {results_count}/{len(motifs)} motifs processed successfully")
        if failed_motifs:
            print(f"❌ Failed motifs: {', '.join(failed_motifs)}")
            print(f"\\n⚠️  CRITICAL: Phase 3c will fail without all 4 motif results")
            print(f"   Recommend re-running failed motifs before Phase 3c:")
            for motif in failed_motifs:
                print(f"     python native_gnn_ablation.py --variant {best_variant} --latent_dim {best_latent} --use-rpb --motif {motif}")

## PHASE 3c: Ablation Strategy Comparison (All Variants) 🔗

Compares SAE latent ablations (Phase 3a) vs native GNN ablations (Phase 3b) for **all 4 SAE variants**:
- Loads ablation results from both strategies
- Computes agreement metrics (Pearson/Spearman correlation) per variant
- **Key Insight:** Identifies which variant has best mechanistic validity (most faithful to GNN)
- High correlation (r > 0.8) = SAE assumptions valid for that variant
- Low correlation = mechanistic limitations to explore

**Note:** This differs from Phase 2 (reconstruction-based ranking). Phase 3c validates mechanistic faithfulness, not just reconstruction quality. You may find that a variant with worse reconstruction actually better explains the GNN's behavior.

**DEPENDS ON:** Both Phase 3a and 3b outputs

**Time:** ~15 minutes

In [ ]:
#@title 7. PHASE 3c: Ablation Strategy Comparison (Motif-Grouped, Single Variant) (~15 min)

import subprocess
import sys
from pathlib import Path
import json

print("\\n" + "="*80)
print("PHASE 3c: ABLATION STRATEGY COMPARISON (MOTIF-GROUPED, SINGLE BEST VARIANT)")
print("="*80)

script = Path('compare_ablation_strategies.py')

if not script.exists():
    print(f"\\n❌ ERROR: {script} not found!")
else:
    print(f"\\n✓ Found comparison script: {script}")

    # Get best variant from Phase 3a metadata
    metadata_file = Path('ablations/phase_3a_config.json')
    if not metadata_file.exists():
        print(f"\\n❌ ERROR: Phase 3a metadata not found!")
        print(f"   Please run Phase 3a first to generate: {metadata_file}")
        sys.exit(1)

    with open(metadata_file, 'r') as f:
        metadata = json.load(f)

    best_config = metadata['best_config']
    best_variant = best_config['variant']
    best_latent = best_config['latent_dim']

    print(f"\\n✓ Using best variant from Phase 3a: {best_variant.upper()}")
    print(f"   Latent Dim: {best_latent}")

    # Check prerequisites
    print(f"\\n📋 Checking Phase 3a and 3b outputs...")

    # Phase 3a check - look for grouped motif results for this variant
    phase_3a_results = []
    if Path('ablations/results').exists():
        phase_3a_results = list(Path('ablations/results').glob(f'*_{best_variant}_l{best_latent}_k*_results.csv'))
        if not phase_3a_results:
            # Fall back to old format (without variant)
            phase_3a_results = list(Path('ablations/results').glob(f'*_l{best_latent}_k*_results.csv'))

    # Phase 3b check - look for motif-specific native ablation files for this variant
    phase_3b_results = []
    if Path('outputs/native_gnn_ablations').exists():
        phase_3b_results = list(Path('outputs/native_gnn_ablations').glob(f'native_ablation_{best_variant}_rpb_*.csv'))

    print(f"   Phase 3a (SAE latent motif groups for {best_variant}): {len(phase_3a_results)} result file(s)")
    if phase_3a_results:
        for f in phase_3a_results[:3]:
            print(f"      ✓ {f.name}")
        if len(phase_3a_results) > 3:
            print(f"      ... and {len(phase_3a_results) - 3} more")

    print(f"   Phase 3b (Native GNN motif groups for {best_variant}): {len(phase_3b_results)} result file(s)")
    if phase_3b_results:
        for f in phase_3b_results[:3]:
            print(f"      ✓ {f.name}")
        if len(phase_3b_results) > 3:
            print(f"      ... and {len(phase_3b_results) - 3} more")

    if len(phase_3a_results) == 0:
        print(f"\\n❌ MISSING: Phase 3a outputs for {best_variant}")

    if len(phase_3b_results) == 0:
        print(f"❌ MISSING: Phase 3b outputs for {best_variant}")

    if len(phase_3a_results) > 0 and len(phase_3b_results) > 0:
        print(f"\\n✓ Both Phase 3a and 3b outputs found")

        print(f"\\n🔗 Comparing SAE latent vs native GNN ablations (MOTIF-GROUPED)...")
        print(f"   Analysis: Motif-group level agreement")
        print(f"   Variant: {best_variant.upper()}")
        print(f"   - Phase 3a: Groups features by motif, ablates groups")
        print(f"   - Phase 3b: Ranks features per motif via r_pb, patches top nodes")
        print(f"   - Phase 3c: Validates agreement between strategies AT MOTIF-GROUP LEVEL")
        print(f"   ")
        print(f"   All 4 motifs: in_feedback_loop, in_cascade, in_feedforward_loop, in_single_input_module")
        print(f"   ")
        print(f"   Output: Motif-group agreement scores + visualizations\\n")

        result = subprocess.run(
            [sys.executable, str(script),
             '--variant', best_variant,   # Use SINGLE best variant (not all-variants)
             '--latent_dim', str(best_latent),
             '--motif-mode'],             # Use MOTIF-GROUP mode
            capture_output=False
        )

        if result.returncode == 0:
            print(f"\\n\\n{'='*80}")
            print(f"✅ PHASE 3c COMPLETED SUCCESSFULLY!")
            print(f"{'='*80}")

            # Check outputs
            comparison_dir = Path('outputs/ablation_strategy_comparison')
            if comparison_dir.exists():
                csvs = list(comparison_dir.glob('*.csv'))
                plots = list(comparison_dir.glob('*.png'))
                print(f"\\n📊 Strategy Comparison Results (Motif-Grouped, {best_variant.upper()}):")
                print(f"   CSV files: {len(csvs)}")
                for csv in sorted(csvs)[:5]:
                    print(f"      ✓ {csv.name}")
                if len(csvs) > 5:
                    print(f"      ... and {len(csvs) - 5} more")

                print(f"   Plots: {len(plots)}")

                # Load and show results
                try:
                    import pandas as pd
                    comp_csv = comparison_dir / 'motif_agreement_summary.csv'
                    if comp_csv.exists():
                        df = pd.read_csv(comp_csv)
                        print(f"\\n🎯 MECHANISTIC VALIDITY FOR {best_variant.upper()}:")
                        print(f"{'─'*80}")
                        print("\\nAgreement by Motif:")
                        for motif in sorted(df['motif'].unique()):
                            motif_data = df[df['motif'] == motif]
                            if len(motif_data) > 0:
                                spearman_r = motif_data['spearman_r'].values[0]
                                n_graphs = int(motif_data['n_graphs'].values[0])
                                print(f"  {motif:25} | ρ = {spearman_r:.3f} ({n_graphs} graphs)")

                        # Overall summary
                        mean_r = df['spearman_r'].mean()
                        print(f"\\n  ✓ Average agreement: ρ = {mean_r:.3f}")
                    else:
                        print(f"\\n   (Could not load detailed results: {comp_csv.name} not found)")
                except Exception as e:
                    print(f"\\n   (Error loading results: {e})")

            print(f"\\n✓ Results saved to: outputs/ablation_strategy_comparison/")
            print(f"\\n→ Continue to PHASE 3d")
        else:
            print(f"\\n❌ Phase 3c failed with return code {result.returncode}")
    else:
        print(f"\\n❌ CANNOT RUN PHASE 3c - MISSING PREREQUISITES:")
        if len(phase_3a_results) == 0:
            print(f"   ✗ Phase 3a outputs for {best_variant}")
            print(f"     Expected: ablations/results/motif_*{best_variant}_l{best_latent}_k*_results.csv")
        if len(phase_3b_results) == 0:
            print(f"   ✗ Phase 3b outputs for {best_variant}")
            print(f"     Expected: outputs/native_gnn_ablations/native_ablation_{best_variant}_rpb_*.csv")
        print(f"\\n   → Complete Phase 3a and 3b first")

## PHASE 3d: Mixed-Motif Generalization Test 🧪

**REQUIRED FOR PUBLICATION:** Validates that features identified in single-motif context generalize to realistic mixed-motif graphs (2-3 interacting motifs):

1. **Preprocessing (one-time):** Generate GNN activations for mixed-motif graphs
2. **Causal Testing:** Test if single-motif-discovered features still cause effects on mixed-motif graphs
3. **Robustness Assessment:** Compare ablation impacts: single-motif vs mixed-motif

**Selection Metric (Default):** Highest max_rpb_abs (strongest feature-motif effect sizes)
- This ensures we test the config with the strongest correlations
- Optional: Use `python compare_sae_configs.py --metric composite_score` to rank by composite_score instead


**Key Question:** Do SAE-learned features work when motifs interact?

**Time:** ~45 minutes

In [ ]:
#@title 8. PHASE 3d: Mixed-Motif Generalization Test (~45 min)

import subprocess
import sys
from pathlib import Path
import pandas as pd
import json

print("\\n" + "="*80)
print("PHASE 3d: MIXED-MOTIF GENERALIZATION TEST")
print("="*80)

print("\\n📋 Step 0: Load Phase 3a config metadata")

# Get best variant from Phase 3a metadata
metadata_file = Path('ablations/phase_3a_config.json')
if not metadata_file.exists():
    print(f"❌ ERROR: Phase 3a metadata not found!")
    print(f"   Please run Phase 3a first to generate: {metadata_file}")
    sys.exit(1)

with open(metadata_file, 'r') as f:
    metadata = json.load(f)

best_config = metadata['best_config']
best_variant = best_config['variant']
best_latent = best_config['latent_dim']

print(f"\\n✓ Using best config from Phase 3a:")
print(f"   Variant: {best_variant}")
print(f"   Latent Dim: {best_latent}")

# Extract variant-specific parameters
if best_variant == 'topk':
    best_k = int(best_config['k'])
    print(f"   K: {best_k}")
elif best_variant == 'gated':
    best_k = int(best_config.get('k', 8))
    print(f"   Sparsity Coef: {best_config['sparsity_coef']:.0e}")
elif best_variant == 'jumprelu':
    best_k = 8
    print(f"   Threshold: {best_config['threshold_init']:.0e}")
elif best_variant == 'switch':
    best_k = int(best_config['k_per_expert'])
    print(f"   Num Experts: {best_config['num_experts']}")
    print(f"   Latent per Expert: {best_config['latent_per_expert']}")
    print(f"   K per Expert: {best_config['k_per_expert']}")

print(f"\\n📋 Step 1: Preprocessing - Generate Mixed-Motif Activations")

script = Path('generate_mixed_motif_activations.py')
if not script.exists():
    print(f"⚠️  {script} not found, skipping preprocessing")
else:
    print(f"Running: python {script}")
    result = subprocess.run([sys.executable, str(script)], capture_output=False)
    if result.returncode == 0:
        print(f"✓ Mixed-motif preprocessing completed")
    else:
        print(f"⚠️  Preprocessing failed, continuing anyway...")

print(f"\\n📋 Step 2: SAE Latent Ablation on Mixed-Motif Graphs")

# Load Phase 2 data to get feature list
csv_file = Path('outputs/latent_correlations.csv')
if not csv_file.exists():
    print(f"❌ ERROR: {csv_file} not found!")
    print(f"   Phase 2 (compare_sae_configs.py) must complete first")
    sys.exit(1)

df_corr = pd.read_csv(csv_file)

# Filter to best variant correlations
df_variant_corr = df_corr[df_corr['variant'] == best_variant]

if df_variant_corr.empty:
    print(f"❌ ERROR: No correlations found for variant {best_variant}")
    sys.exit(1)

# Get top features (by |rpb|) from each motif
motif_features = {}
for motif in ['in_cascade', 'in_feedback_loop', 'in_feedforward_loop', 'in_single_input_module']:
    top_features = df_variant_corr[df_variant_corr['motif'] == motif].nlargest(5, 'rpb_abs')['feature'].tolist()
    motif_features[motif] = top_features

# Combine all features
all_features = []
for features in motif_features.values():
    all_features.extend(features)
all_features = sorted(list(set(all_features)))  # Remove duplicates

print(f"\\n✓ Selected top features across all motifs: {len(all_features)} features")
feature_str = ','.join([str(f) for f in all_features])

# Run SAE ablation on mixed-motif graphs
ablation_script = Path('run_ablation.py')
if not ablation_script.exists():
    print(f"❌ ERROR: {ablation_script} not found!")
else:
    print(f"\\nRunning: run_ablation.py on mixed-motif data with top features")
    print(f"  Command: python run_ablation.py --variant {best_variant} --latent_dim {best_latent} --use_mixed_motifs --feature {feature_str}")

    result = subprocess.run([
        sys.executable, str(ablation_script),
        '--variant', best_variant,
        '--latent_dim', str(best_latent),
        '--use_mixed_motifs',
        '--feature', feature_str,
        '--experiment_name', f'mixed_motifs_{best_variant}'
    ], capture_output=False)

    if result.returncode == 0:
        print(f"\\n✓ SAE mixed-motif ablation completed")
    else:
        print(f"\\n⚠️  SAE mixed-motif ablation failed")

print(f"\\n📋 Step 3: Native GNN Ablation on Mixed-Motif Graphs")

native_script = Path('native_gnn_ablation.py')
if not native_script.exists():
    print(f"❌ ERROR: {native_script} not found!")
else:
    print(f"Running: native_gnn_ablation.py on mixed-motif data with top features")
    print(f"  Command: python native_gnn_ablation.py --variant {best_variant} --latent_dim {best_latent} --use_mixed_motifs --feature {feature_str}")

    result = subprocess.run([
        sys.executable, str(native_script),
        '--variant', best_variant,
        '--latent_dim', str(best_latent),
        '--use_mixed_motifs',
        '--feature', feature_str
    ], capture_output=False)

    if result.returncode == 0:
        print(f"\\n✓ Native mixed-motif ablation completed")
    else:
        print(f"\\n⚠️  Native mixed-motif ablation failed")

print(f"\\n\\n{'='*80}")
print(f"✅ PHASE 3d COMPLETED!")
print(f"{'='*80}")
print(f"\\nMixed-motif generalization test results:")
print(f"  SAE ablation: ablations/results/ablation_*_mixed_motifs.csv")
print(f"  Native ablation: outputs/native_gnn_ablations/native_ablation_*_mixed_motifs.csv")
print(f"\\n→ Continue to PHASE 4 (Statistical Analysis)")

## PHASE 4: Statistical Validation 📈

Comprehensive statistical analysis:
- Feature-motif correlation distributions (per variant, per motif)
- Feature redundancy analysis (decoder similarity)
- Ablation effects conditioned on motif presence (Wilcoxon test)
- Sparsity-interpretability trade-off curves
- FDR-corrected statistical significance

**Time:** ~30 minutes

In [ ]:
#@title 8. PHASE 4: Statistical Validation (~30 min)

import subprocess
import sys
from pathlib import Path
import json
import pandas as pd

print("\n" + "="*80)
print("PHASE 4: STATISTICAL VALIDATION")
print("="*80)

script = Path('statistical_analysis_suite.py')

if not script.exists():
    print(f"\n❌ ERROR: {script} not found!")
else:
    print(f"\n✓ Found analysis script: {script}")

    # Check prerequisites
    print(f"\n📋 Checking Phase 2b prerequisites (multi-seed models)...")
    multi_seed_ckpts = list(Path('checkpoints').glob('sae_*_seed*.pt'))
    
    if len(multi_seed_ckpts) == 0:
        print(f"   ⚠️  WARNING: No multi-seed checkpoints found!")
        print(f"   Expected: checkpoints/sae_*_seed*.pt")
        print(f"   CRITICAL: Phase 2b (retrain_best_configs.py) must complete first")
        print(f"   Limited analysis will be performed without multi-seed stability data")
        run_seed_analysis = False
    else:
        total_seeds = len(multi_seed_ckpts)
        print(f"   ✓ Found {total_seeds} multi-seed checkpoints")
        print(f"   ✓ Ready for feature stability analysis across seeds")
        run_seed_analysis = True

    print(f"\n📊 Running comprehensive statistical analysis...")
    print(f"   Analysis 1: Correlation distributions per variant & motif")
    print(f"   Analysis 2: Feature redundancy analysis (decoder similarity)")
    print(f"   Analysis 3: Ablation conditional effects (Wilcoxon test)")
    print(f"   Analysis 4: Sparsity-interpretability trade-off")
    
    if run_seed_analysis:
        print(f"   Analysis 5: Feature stability across seeds (multi-seed)")
    else:
        print(f"   Analysis 5: SKIPPED (requires multi-seed models from Phase 2b)")
    
    print(f"\n")

    # Build argument list
    args = [sys.executable, str(script),
            '--variant', 'all',
            '--redundancy',
            '--tradeoff']
    
    if run_seed_analysis:
        args.append('--seed-analysis')
    
    result = subprocess.run(args, capture_output=False)

    if result.returncode == 0:
        print(f"\n\n{'='*80}")
        print(f"✅ PHASE 4 COMPLETED SUCCESSFULLY!")
        print(f"{'='*80}")

        # Show generated outputs
        stats_dir = Path('outputs/statistical_analysis')
        if stats_dir.exists():
            plots = list(stats_dir.glob('*.png'))
            reports = list(stats_dir.glob('*.md'))
            csvs = list(stats_dir.glob('*.csv'))

            print(f"\n📊 Generated Outputs:")
            print(f"   Visualizations: {len(plots)} plots")
            print(f"   Reports: {len(reports)}")
            print(f"   Data: {len(csvs)} CSV files")

            if plots:
                print(f"\n   Sample Plots:")
                for plot in sorted(plots)[:3]:
                    print(f"      ✓ {plot.name}")

        # Check if stability analysis was performed
        if run_seed_analysis:
            stability_plot = stats_dir / 'feature_stability.png'
            if stability_plot.exists():
                print(f"\n   ✓ Feature Stability Analysis Complete")
                print(f"      Output: feature_stability.png (multi-seed cross-seed similarity)")
        else:
            print(f"\n   ⚠️  Feature Stability Analysis SKIPPED (requires Phase 2b)")
            print(f"      To enable: Complete Phase 2b (retrain_best_configs.py)")

        print(f"\n✓ Results saved to: outputs/statistical_analysis/")
        print(f"\n→ Continue to PHASE 5")
    else:
        print(f"\n❌ Phase 4 failed with return code {result.returncode}")


## PHASE 5: Visualization & Feature Analysis 🎨

Feature selectivity analysis and visualization:
- Feature activation selectivity heatmaps  
- Feature importance rankings
- Motif-feature correlation patterns

**Note:** Reconstruction fidelity analysis moved to Phase 2.5b (analyzes all variants)

**Time:** ~15 minutes

In [ ]:
#@title 9. PHASE 5: Visualization & Feature Analysis (~15 min)

import subprocess
import sys
from pathlib import Path

print("\n" + "="*80)
print("PHASE 5: VISUALIZATION & FEATURE ANALYSIS")
print("="*80)

# Phase 5: Feature visualization
script_viz = Path('visualize_feature_activations.py')

if not script_viz.exists():
    print(f"\n⚠️  Visualization script not found:")
    print(f"   - {script_viz}")
else:
    print(f"\n✓ Found visualization script: {script_viz}")

    # Get best config from Phase 2 for feature viz
    import pandas as pd
    csv_file = Path('outputs/sae_config_comparison.csv')
    if csv_file.exists():
        df = pd.read_csv(csv_file)
        best = df.iloc[0]  # Best overall config
        best_variant = best['variant']
        best_latent = int(best['latent_dim'])
        best_k = int(best.get('k', 8))

        print(f"\n✓ Using best config from Phase 2:")
        print(f"   Variant: {best_variant}")
        print(f"   Latent Dim: {best_latent}")
        
        print(f"\n📊 Part 1: Feature Activation Visualization")
        print(f"{'─'*80}\n")

        result = subprocess.run(
            [sys.executable, str(script_viz),
             '--variant', best_variant,
             '--latent_dim', str(best_latent),
             '--features', '20'],
            capture_output=False
        )

        if result.returncode == 0:
            print(f"\n✓ Feature visualization completed")
        else:
            print(f"\n⚠️  Feature visualization failed (non-critical)")
    else:
        print(f"\n⚠️  Could not find Phase 2 results (sae_config_comparison.csv)")
        print(f"   Skipping Phase 5 (Phase 2 must complete first)")

    print(f"\n{'─'*80}")
    print(f"📊 Reconstruction Fidelity Analysis (PCA Histograms)")
    print(f"{'─'*80}")
    print(f"✓ Moved to PHASE 2.5b (analyzes all 4 variants' best configs)")
    print(f"   Results: outputs/sae_reconstruction_fidelity/pca_histograms_*.png\n")

    print(f"\n{'='*80}")
    print(f"✅ PHASE 5 COMPLETED!")
    print(f"{'='*80}")

    # List visualization outputs
    viz_dirs = [
        ('Feature Activations', Path('outputs/feature_activation_visualizations')),
        ('Reconstruction Fidelity (Phase 2.5b)', Path('outputs/sae_reconstruction_fidelity'))
    ]

    print(f"\n📈 Generated Visualizations:")
    for name, viz_dir in viz_dirs:
        if viz_dir.exists():
            plots = list(viz_dir.glob('*.png'))
            status = "✓" if plots else "⏳"
            print(f"   {status} {name}: {len(plots)} plots")

    print(f"\n→ Continue to Final Summary & Export")

## Final Summary & Export Results 💾

In [ ]:
#@title 10. Final Results Summary

import pandas as pd
import json
from pathlib import Path

print("\n" + "="*80)
print("FINAL PIPELINE SUMMARY")
print("="*80)

# Check all phases completed
phases_status = {
    'Phase 1 (Training)': len(list(Path('checkpoints').glob('sae_*.pt'))) >= 25,
    'Phase 2 (Configuration Comparison)': Path('outputs/sae_config_comparison.csv').exists(),
    'Phase 3a (SAE Latent Ablations)': Path('ablations/results').exists() and len(list(Path('ablations/results').glob('*.csv'))) > 0,
    'Phase 3b (Native GNN Ablations)': Path('outputs/native_gnn_ablations').exists() and len(list(Path('outputs/native_gnn_ablations').glob('*.csv'))) > 0,
    'Phase 3c (Strategy Comparison)': Path('outputs/ablation_strategy_comparison').exists(),
    'Phase 4 (Statistical Analysis)': Path('outputs/statistical_analysis').exists(),
    'Phase 5 (Visualization)': Path('outputs/sae_reconstruction_fidelity').exists() or Path('outputs/feature_activation_visualizations').exists()
}

print(f"\n📋 Pipeline Status:")
completed = 0
for phase, status in phases_status.items():
    symbol = "✅" if status else "⏳"
    print(f"   {symbol} {phase}")
    if status:
        completed += 1

print(f"\n📊 Overall Progress: {completed}/{len(phases_status)} phases")

# Load and display results
csv_file = Path('outputs/sae_config_comparison.csv')
if csv_file.exists():
    df = pd.read_csv(csv_file)

    print(f"\n📊 Training Results:")
    print(f"   Total Configurations: {len(df)}")
    print(f"\n   Per-Variant:")
    for variant in ['topk', 'gated', 'jumprelu', 'switch']:
        v_data = df[df['variant'] == variant]
        if len(v_data) > 0:
            print(f"      {variant.upper()}: {len(v_data)} configs")

    # Best configuration
    if 'composite_score' in df.columns:
        best_idx = df['composite_score'].idxmax()
        best = df.loc[best_idx]
        print(f"\n🏆 Best Overall Configuration:")
        print(f"   Variant: {best['variant'].upper()}")
        print(f"   Config: {best['config_name']}")
        if 'composite_score' in best:
            print(f"   Composite Score: {best['composite_score']:.3f}")

print(f"\n" + "="*80)
if completed == len(phases_status):
    print(f"✅ COMPLETE ANALYSIS PIPELINE FINISHED!")
else:
    print(f"⏳ Pipeline in progress ({completed}/{len(phases_status)} phases complete)")
print(f"="*80)
print(f"\n→ See next cell to export results to Google Drive")

In [ ]:
#@title 11. Export All Results to Google Drive

import shutil
from pathlib import Path

print("\n" + "="*80)
print("EXPORTING ALL RESULTS TO GOOGLE DRIVE")
print("="*80)

DRIVE_RESULTS = Path('/content/drive/MyDrive/SAE_Results_Output')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Destination: {DRIVE_RESULTS}")

# Directories to export
export_dirs = [
    ('checkpoints', 'Trained SAE Model Checkpoints (30 files)'),
    ('outputs', 'Analysis Outputs (CSV, plots, reports)'),
    ('ablations', 'Ablation Study Results')
]

print(f"\n💾 Copying Results:")

for src_name, description in export_dirs:
    src_dir = Path(src_name)
    if src_dir.exists():
        dst_dir = DRIVE_RESULTS / src_name
        print(f"\n   {description}...", end=' ')

        if dst_dir.exists():
            shutil.rmtree(dst_dir)

        shutil.copytree(src_dir, dst_dir)

        # Count files
        file_count = sum(1 for _ in dst_dir.rglob('*') if _.is_file())
        print(f"✓ ({file_count} files)")
    else:
        print(f"\n   {description}... ⚠️  (not found, skipping)")

print(f"\n" + "="*80)
print(f"✅ EXPORT COMPLETE!")
print(f"="*80)
print(f"\n📁 Results Location: {DRIVE_RESULTS}")
print(f"\nYou can now:")
print(f"  1. ✓ Download all files to your computer")
print(f"  2. ✓ Use outputs in your research")
print(f"  3. ✓ Share with collaborators")
print(f"  4. ✓ Run further analysis locally if needed")

In [ ]:
#@title 12. Next Steps & Resources

print("""
╔════════════════════════════════════════════════════════════════════════════════╗
║                    PIPELINE EXECUTION COMPLETE!                                 ║
╚════════════════════════════════════════════════════════════════════════════════╝

You now have:

1. 📊 30 Trained SAE Models
   - TopK SAE (11 configs)
   - Gated SAE (9 configs)
   - JumpReLU SAE (6 configs)
   - Switch SAE (4 configs)

2. 📈 Complete Analysis Pipeline Results
   ✓ Phase 1: SAE Training
   ✓ Phase 2: Configuration Comparison
   ✓ Phase 3a: SAE Latent Ablations
   ✓ Phase 3b: Native GNN Ablations
   ✓ Phase 3c: Strategy Comparison
   ✓ Phase 4: Statistical Validation
   ✓ Phase 5: Visualization & Reconstruction Analysis

3. 📊 Publication-Ready Outputs
   - Comparison tables (CSV)
   - Statistical analysis plots
   - Ablation results
   - Feature visualizations

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📌 KEY OUTPUTS:

Checkpoints/:
  - sae_topk_latent512_k8_seed42.pt          (TopK best)
  - sae_gated_latent512_lambda1e-03_seed42.pt (Gated best)
  - etc... (30 total)

Outputs/:
  - sae_config_comparison.csv                (within-variant ranking)
  - statistical_analysis/                    (plots + reports)
  - native_gnn_ablations/                    (direct patching results)
  - ablation_strategy_comparison/            (agreement metrics)
  - sae_reconstruction_fidelity/             (PCA histograms)

Ablations/:
  - results/ablation_*.csv                   (phase 3a outputs)
  - plots/ablation_*.png                     (visualizations)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 WHAT'S DIFFERENT FROM BEFORE?

The fixed notebook now:
  ✅ Runs phases in correct order (1 → 2 → 3a → 3b → 3c → 4 → 5)
  ✅ Includes missing Phase 3a (run_ablation.py execution)
  ✅ Verifies data prerequisites before each phase
  ✅ Explicitly checks for required outputs (test_graph_ids.json, ablation CSVs)
  ✅ Clear phase labels matching official pipeline
  ✅ Data dependency documentation

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📚 REFERENCE DOCUMENTS:

In your project folder:
  - INTERPRETABILITY_PIPELINE_GUIDE.md     (complete pipeline architecture)
  - SAE_COLAB_PIPELINE_AUDIT.md            (audit report for this notebook)
  - ANALYSIS_WORKFLOW.md                   (step-by-step guide)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎉 You're all set! The pipeline is ready to run.
""")